# Angular-Quadrature Convergence of Slab Leakage

This tutorial repeats an absorbing-slab calculation with increasingly resolved Gauss-Legendre quadratures and compares the transmitted current with an analytic reference.

**Audience:** Users selecting angular resolution and assessing discretization error.

**Prerequisites:** Angular quadrature and isotropic boundary sources.

## Repeat the calculation at several angular orders

For a unit-thickness pure absorber with the boundary normalization used below, the transmitted current is approximately 0.109692. Only the number of polar ordinates changes between runs.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.xs import MultiGroupXS

rank = MPI.COMM_WORLD.rank
reference_current = 0.109692

def transmitted_current(n_polar):
    mesh = OrthogonalMeshGenerator(node_sets=[[i / 100.0 for i in range(101)]]).Execute()
    mesh.SetUniformBlockID(0)
    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.0)
    quadrature = GLProductQuadrature1DSlab(n_polar=n_polar, scattering_order=0)
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[{"groups_from_to": (0, 0), "angular_quadrature": quadrature}],
        xs_map=[{"block_ids": [0], "xs": xs}],
        boundary_conditions=[
            {"name": "zmin", "type": "isotropic", "group_strength": [2.0]},
            {"name": "zmax", "type": "vacuum"},
        ],
        options={"save_angular_flux": True},
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()
    return float(problem.ComputeLeakage(["zmax"])["zmax"][0])

orders = [4, 8, 16, 32]
currents = [transmitted_current(order) for order in orders]
errors = [abs(current - reference_current) for current in currents]

## Interpret the error

Increasing angular order should reduce quadrature error until another error source, such as the spatial mesh or the precision of the reference value, dominates.

In [ ]:
if rank == 0:
    for order, current, error in zip(orders, currents, errors):
        print(f"S{order:02d}: current={current:.8e}, error={error:.6e}")
    print(f"Angular-convergence final error={errors[-1]:.6e}")
assert errors[-1] < errors[0]
assert errors[-1] < 1.0e-4
if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()